# Data Gathering and cleaning

### note:- directly use cleaned csv for imports no need to do it again

In [29]:
import pandas as pd
import numpy as np
import glob
import os

In [3]:
%ls

 Volume in drive D is Work
 Volume Serial Number is 869A-DE4F

 Directory of d:\cicids

21-06-2026  09:04    <DIR>          .
21-06-2026  09:04    <DIR>          ..
21-06-2026  09:13             1,923 dataclean.ipynb
21-06-2026  09:04        77,123,859 Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
21-06-2026  09:04        76,906,168 Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
21-06-2026  09:04        58,316,725 Friday-WorkingHours-Morning.pcap_ISCX.csv
21-06-2026  09:04       176,927,918 Monday-WorkingHours.pcap_ISCX.csv
21-06-2026  09:04        83,102,436 Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
21-06-2026  09:04        52,023,263 Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
21-06-2026  09:04       135,078,995 Tuesday-WorkingHours.pcap_ISCX.csv
21-06-2026  09:04       225,166,395 Wednesday-workingHours.pcap_ISCX.csv
               9 File(s)    884,647,682 bytes
               2 Dir(s)  470,859,317,248 bytes free


In [7]:
dataset_path = r'' 
all_files = glob.glob(os.path.join(dataset_path, "*.csv"))
all_files

['Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv',
 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv',
 'Friday-WorkingHours-Morning.pcap_ISCX.csv',
 'Monday-WorkingHours.pcap_ISCX.csv',
 'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv',
 'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
 'Tuesday-WorkingHours.pcap_ISCX.csv',
 'Wednesday-workingHours.pcap_ISCX.csv']

In [8]:
len(all_files)

8

In [12]:
df_list = [pd.read_csv(file) for file in all_files]

In [17]:
df = pd.concat(df_list, ignore_index=True)

In [21]:
df.columns = df.columns.str.strip()

In [23]:
# Attacker IP
kali_attacker = '205.174.165.73'
ddos_attackers = ['205.174.165.69', '205.174.165.70', '205.174.165.71']

# Victim IP
web_server_16 = '192.168.10.50'
ubuntu_server_12 = '192.168.10.51'
win_vista = '192.168.10.8'
win_10_64 = '192.168.10.15'

In [24]:
source_ip_mapping = {
    # Monday
    'BENIGN': 'Benign_Node', 
    
    # Tuesday
    'FTP-Patator': kali_attacker,
    'SSH-Patator': kali_attacker,
    
    # Wednesday
    'DoS slowloris': kali_attacker,
    'DoS Slowhttptest': kali_attacker,
    'DoS Hulk': kali_attacker,
    'DoS GoldenEye': kali_attacker,
    'Heartbleed': kali_attacker,
    
    # Thursday
    'Web Attack  Brute Force': kali_attacker,
    'Web Attack  XSS': kali_attacker,
    'Web Attack  Sql Injection': kali_attacker,
    'Web Attack - Brute Force': kali_attacker,
    'Web Attack - XSS': kali_attacker,
    'Web Attack - Sql Injection': kali_attacker,
    
    # Infiltration
    'Infiltration': kali_attacker, 
    
    # Friday
    'Bot': kali_attacker,
    'PortScan': kali_attacker
}

In [26]:
destination_ip_mapping = {
    # Monday
    'BENIGN': 'Benign_Node', 
    
    # Tuesday
    'FTP-Patator': web_server_16,
    'SSH-Patator': web_server_16,
    
    # Wednesday
    'DoS slowloris': web_server_16,
    'DoS Slowhttptest': web_server_16,
    'DoS Hulk': web_server_16,
    'DoS GoldenEye': web_server_16,
    'Heartbleed': ubuntu_server_12,
    
    # Thursday
    'Web Attack  Brute Force': web_server_16,
    'Web Attack  XSS': web_server_16,
    'Web Attack  Sql Injection': web_server_16,
    'Web Attack - Brute Force': web_server_16,
    'Web Attack - XSS': web_server_16,
    'Web Attack - Sql Injection': web_server_16,
    
    # Thursday Infiltration
    'Infiltration': win_vista,
    
    # Friday
    'Bot': win_10_64,
    'PortScan': web_server_16,
    'DDoS': web_server_16
}

In [27]:
# Labelling ip
df['Source_IP'] = df['Label'].map(source_ip_mapping)
df['Destination_IP'] = df['Label'].map(destination_ip_mapping).fillna('Unknown_IP')

In [30]:
# Handle the multi-attacker DDoS distribution
ddos_mask = df['Label'] == 'DDoS'
# Randomly assign one of the 3 attacker IPs to each row identified as DDoS
df.loc[ddos_mask, 'Source_IP'] = np.random.choice(ddos_attackers, size=ddos_mask.sum())

In [31]:
df['Source_IP'] = df['Source_IP'].fillna('Unknown_IP')

In [32]:
output_filename = 'CICIDS2017_Merged_with_IPs.csv'
df.to_csv(output_filename, index=False)

In [33]:
import gc

In [34]:
try:
    del df
    del df_list
    del ddos_mask
    print("Old DataFrames and variables deleted.")
except NameError:
    print("Variables already cleared.")

Old DataFrames and variables deleted.


In [36]:
gc.collect()

0

In [37]:
df_clean = pd.read_csv('CICIDS2017_Merged_with_IPs.csv', low_memory=False)

In [38]:
df_clean.shape

(2830743, 81)

In [41]:
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns

nan_counts = df_clean.isna().sum()
nan_cols = nan_counts[nan_counts > 0]

# We must check for infs only in numeric columns to avoid string errors
inf_counts = np.isinf(df_clean[numeric_cols]).sum()
inf_cols = inf_counts[inf_counts > 0]

print(f"Columns with NaNs: {len(nan_cols)}")
if len(nan_cols) > 0:
    print(nan_cols.head())

print(f"\nColumns with Infinities: {len(inf_cols)}")
if len(inf_cols) > 0:
    print(inf_cols.head())

Columns with NaNs: 1
Flow Bytes/s    1358
dtype: int64

Columns with Infinities: 2
Flow Bytes/s      1509
Flow Packets/s    2867
dtype: int64


In [42]:
unique_counts = df_clean.nunique()
constant_cols = unique_counts[unique_counts == 1].index.tolist()

print(f"Found {len(constant_cols)} columns with only 1 unique value:")
print(constant_cols)

Found 8 columns with only 1 unique value:
['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']


In [43]:
object_cols = df_clean.select_dtypes(include=['object']).columns.tolist()

print(f"Found {len(object_cols)} non-numeric (string) columns:")
print(object_cols)

Found 3 non-numeric (string) columns:
['Label', 'Source_IP', 'Destination_IP']


C:\Users\potato\AppData\Local\Temp\ipykernel_3244\1435831548.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df_clean.select_dtypes(include=['object']).columns.tolist()


In [45]:
if 'Label' in df_clean.columns:
    dist = df_clean['Label'].value_counts()
    print(dist)
    
    benign_pct = (dist.get('BENIGN', 0) / len(df_clean)) * 100
    print(f"\nOverall Benign Traffic: {benign_pct:.2f}%")
else:
    print("Label column not found.")

Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

Overall Benign Traffic: 80.30%


In [46]:
zero_var_cols = [
    'Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 
    'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 
    'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate'
]
df_clean.drop(columns=zero_var_cols, inplace=True, errors='ignore')
print(f"Dropped {len(zero_var_cols)} zero-variance columns.")

Dropped 8 zero-variance columns.


In [47]:
df_clean.replace([np.inf, -np.inf], np.nan, inplace=True)

missing_before = len(df_clean)
df_clean.dropna(inplace=True)
dropped_rows = missing_before - len(df_clean)

print(f"Dropped {dropped_rows} rows containing NaNs or Infinities.")

Dropped 2867 rows containing NaNs or Infinities.


In [48]:
df_clean['Label'] = df_clean['Label'].str.replace('', '-', regex=False)
df_clean['Label'] = df_clean['Label'].str.replace('Web Attack - ', 'Web Attack - ', regex=False) # Normalize spacing
print("Cleaned Label encoding artifacts.")

Cleaned Label encoding artifacts.


In [49]:
df_clean.shape

(2827876, 73)

In [50]:
output_filename = 'CICIDS2017_clean.csv'
df_clean.to_csv(output_filename, index=False)

In [51]:
try:
    del df_clean
    print("Old DataFrame deleted.")
except NameError:
    print("No previous DataFrame found in memory.")

gc.collect()

Old DataFrame deleted.


0

In [52]:
df = pd.read_csv('CICIDS2017_clean.csv', low_memory=False)

In [53]:
df.describe()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
count,2.827876e+06,2.827876e+06,2.827876e+06,2.827876e+06,2.827876e+06,2.827876e+06,2.827876e+06,2.827876e+06,2.827876e+06,2.827876e+06,...,2.827876e+06,2.827876e+06,2.827876e+06,2.827876e+06,2.827876e+06,2.827876e+06,2.827876e+06,2.827876e+06,2.827876e+06,2.827876e+06
mean,8.061534e+03,1.480065e+07,9.368972e+00,1.040396e+01,5.498522e+02,1.617903e+04,2.078044e+02,1.872929e+01,5.825628e+01,6.897811e+01,...,5.423519e+00,-2.744494e+03,8.163400e+04,4.117582e+04,1.533378e+05,5.835492e+04,8.324468e+06,5.043548e+05,8.704568e+06,7.928061e+06
std,1.827432e+04,3.366750e+07,7.500527e+02,9.978937e+02,9.998639e+03,2.264235e+06,7.175183e+02,6.035533e+01,1.861733e+02,2.813212e+02,...,6.367482e+02,1.085539e+06,6.489234e+05,3.935787e+05,1.026333e+06,5.773818e+05,2.364057e+07,4.605289e+06,2.437766e+07,2.337390e+07
min,0.000000e+00,-1.300000e+01,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,-5.368707e+08,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,5.300000e+01,1.550000e+02,2.000000e+00,1.000000e+00,1.200000e+01,2.000000e+00,6.000000e+00,0.000000e+00,6.000000e+00,0.000000e+00,...,0.000000e+00,2.000000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,8.000000e+01,3.133800e+04,2.000000e+00,2.000000e+00,6.200000e+01,1.230000e+02,3.700000e+01,2.000000e+00,3.400000e+01,0.000000e+00,...,1.000000e+00,2.400000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,4.430000e+02,3.239368e+06,5.000000e+00,4.000000e+00,1.880000e+02,4.840000e+02,8.100000e+01,3.600000e+01,5.000000e+01,2.616295e+01,...,2.000000e+00,3.200000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,6.553500e+04,1.200000e+08,2.197590e+05,2.919220e+05,1.290000e+07,6.554530e+08,2.482000e+04,2.325000e+03,5.940857e+03,7.125597e+03,...,2.135570e+05,1.380000e+02,1.100000e+08,7.420000e+07,1.100000e+08,1.100000e+08,1.200000e+08,7.690000e+07,1.200000e+08,1.200000e+08


In [71]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2827876 entries, 0 to 2827875
Data columns (total 73 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Destination Port             int64  
 1   Flow Duration                int64  
 2   Total Fwd Packets            int64  
 3   Total Backward Packets       int64  
 4   Total Length of Fwd Packets  int64  
 5   Total Length of Bwd Packets  int64  
 6   Fwd Packet Length Max        int64  
 7   Fwd Packet Length Min        int64  
 8   Fwd Packet Length Mean       float64
 9   Fwd Packet Length Std        float64
 10  Bwd Packet Length Max        int64  
 11  Bwd Packet Length Min        int64  
 12  Bwd Packet Length Mean       float64
 13  Bwd Packet Length Std        float64
 14  Flow Bytes/s                 float64
 15  Flow Packets/s               float64
 16  Flow IAT Mean                float64
 17  Flow IAT Std                 float64
 18  Flow IAT Max                 int64  
 19  Flow IAT Mi

In [67]:
print(df['Destination_IP'].value_counts(), '\n')
print(df['Source_IP'].value_counts(), '\n')
print(df['Label'].value_counts(), '\n')


Destination_IP
Benign_Node      2271320
192.168.10.50     552373
Unknown_IP          2180
192.168.10.15       1956
192.168.10.8          36
192.168.10.51         11
Name: count, dtype: int64 

Source_IP
Benign_Node       2271320
205.174.165.73     426351
205.174.165.69      42806
205.174.165.71      42718
205.174.165.70      42501
Unknown_IP           2180
Name: count, dtype: int64 

Label
BENIGN                        2271320
DoS Hulk                       230124
PortScan                       158804
DDoS                           128025
DoS GoldenEye                   10293
FTPPatator                       7935
SSHPatator                       5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1956
Web Attack - Brute Force         1507
Web Attack - XSS                  652
Infiltration                       36
Web Attack - Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64 



# Charts

In [72]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [75]:
output_dir = 'data_analysis'
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# df = pd.read_csv('CICIDS2017_Fully_Cleaned_For_GNN.csv', low_memory=False)

In [76]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Found {len(numeric_cols)} numeric features for analysis.")

Found 70 numeric features for analysis.


In [77]:
df['Binary_Label'] = df['Label'].apply(lambda x: 'Benign' if x == 'BENIGN' else 'Attack')

In [79]:
plt.figure(figsize=(8, 6))
ax1 = sns.countplot(data=df, x='Binary_Label', palette=['#2ecc71', '#e74c3c'])
plt.title('Binary Classification: Benign vs Attack Traffic')
plt.ylabel('Number of Flows')

# Add exact numeric labels to the top of the bars
for container in ax1.containers:
    ax1.bar_label(container, fmt='{:,.0f}', padding=5, fontsize=11, fontweight='bold')

plt.savefig(f'{output_dir}/01_binary_distribution.png', dpi=300, bbox_inches='tight')
plt.close()

C:\Users\potato\AppData\Local\Temp\ipykernel_3244\2533189674.py:2: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax1 = sns.countplot(data=df, x='Binary_Label', palette=['#2ecc71', '#e74c3c'])


In [81]:
plt.figure(figsize=(14, 10))
ax2 = sns.countplot(data=df, y='Label', order=df['Label'].value_counts().index, palette='viridis')
plt.xscale('log')
plt.title('Class-Wise Traffic Distribution (Log Scale)')
plt.xlabel('Number of Flows (Log Scale)')
plt.ylabel('Traffic Label')

# Add exact numeric labels to the end of the horizontal bars
for container in ax2.containers:
    ax2.bar_label(container, fmt=' {:,.0f}', padding=3, fontsize=10)

plt.savefig(f'{output_dir}/02_class_distribution_log.png', dpi=300, bbox_inches='tight')
plt.close()

C:\Users\potato\AppData\Local\Temp\ipykernel_3244\1462068059.py:2: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  ax2 = sns.countplot(data=df, y='Label', order=df['Label'].value_counts().index, palette='viridis')


In [82]:
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(24, 20))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
# annot=False remains because 4,900 overlapping text boxes will ruin the chart
sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', center=0, 
            annot=False, linewidths=0.5, cbar_kws={"shrink": .5})
plt.title('Feature Correlation Matrix')
plt.savefig(f'{output_dir}/03_correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.close()

In [87]:
plt.figure(figsize=(16, 10))
sns.boxplot(data=df, x='Binary_Label', y='Flow Duration', palette='Set2')
plt.yscale('log')
plt.title('Boxplot Density: Flow Duration by Class (Notice the extreme scale)')
plt.savefig(f'{output_dir}/04_boxplot_flow_duration.png', dpi=300, bbox_inches='tight')
plt.close()

C:\Users\potato\AppData\Local\Temp\ipykernel_3244\1421655636.py:2: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x='Binary_Label', y='Flow Duration', palette='Set2')


In [1]:
import pandas as pd
import numpy as np
import gc
from sklearn.preprocessing import StandardScaler

In [14]:
try:
    del df, sample_df, benign_df, attack_df, corr_matrix, mask, pairplot_fig
    print("Old EDA variables deleted.")
except NameError:
    print("No previous EDA variables found.")

No previous EDA variables found.


In [15]:
gc.collect()

2463

In [25]:
df = pd.read_csv('CICIDS2017_clean.csv', low_memory=False)
print(f"Dataset loaded. Initial shape: {df.shape}\n")

Dataset loaded. Initial shape: (2827876, 73)



In [26]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df[numeric_cols].corr().abs()

In [27]:
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Find features with correlation greater than 0.95
thresholds = [0.95, 0.96, 0.97, 0.98, 0.99, 0.995]
for threshold in thresholds:
    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
    print(threshold, " - ", len(to_drop))
    to_drop.sort()
    print(f"Found {(to_drop)} highly correlated features (>{threshold}):")

0.95  -  23
Found ['Average Packet Size', 'Avg Bwd Segment Size', 'Avg Fwd Segment Size', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'CWE Flag Count', 'ECE Flag Count', 'Fwd Header Length.1', 'Fwd IAT Max', 'Fwd IAT Total', 'Fwd Packet Length Std', 'Fwd Packets/s', 'Idle Max', 'Idle Mean', 'Idle Min', 'Packet Length Std', 'SYN Flag Count', 'Subflow Bwd Bytes', 'Subflow Bwd Packets', 'Subflow Fwd Bytes', 'Subflow Fwd Packets', 'Total Backward Packets', 'Total Length of Bwd Packets'] highly correlated features (>0.95):
0.96  -  22
Found ['Average Packet Size', 'Avg Bwd Segment Size', 'Avg Fwd Segment Size', 'Bwd Packet Length Std', 'CWE Flag Count', 'ECE Flag Count', 'Fwd Header Length.1', 'Fwd IAT Max', 'Fwd IAT Total', 'Fwd Packet Length Std', 'Fwd Packets/s', 'Idle Max', 'Idle Mean', 'Idle Min', 'Packet Length Std', 'SYN Flag Count', 'Subflow Bwd Bytes', 'Subflow Bwd Packets', 'Subflow Fwd Bytes', 'Subflow Fwd Packets', 'Total Backward Packets', 'Total Length of Bwd Packets'] 

In [28]:
threshold = 0.99
to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
print(f"Found {len(to_drop)} highly correlated features (>{threshold}):")
for col in to_drop:
    print(f" - Dropping: {col}")

# Drop the highly correlated features
df.drop(columns=to_drop, inplace=True)
print(f"\nShape after removing collinearity: {df.shape}\n")

Found 17 highly correlated features (>0.99):
 - Dropping: Total Backward Packets
 - Dropping: Total Length of Bwd Packets
 - Dropping: Fwd IAT Total
 - Dropping: Fwd IAT Max
 - Dropping: SYN Flag Count
 - Dropping: CWE Flag Count
 - Dropping: ECE Flag Count
 - Dropping: Average Packet Size
 - Dropping: Avg Fwd Segment Size
 - Dropping: Avg Bwd Segment Size
 - Dropping: Fwd Header Length.1
 - Dropping: Subflow Fwd Packets
 - Dropping: Subflow Fwd Bytes
 - Dropping: Subflow Bwd Packets
 - Dropping: Subflow Bwd Bytes
 - Dropping: Idle Max
 - Dropping: Idle Min

Shape after removing collinearity: (2827876, 56)



In [29]:
print("--- 5. SAVING ML-READY DATASET No scaling ---")
output_filename = 'CICIDS2017_ML_Ready_NoScale.csv'
df.to_csv(output_filename, index=False)

print(f"Success! Final ML-Ready dataset saved to {output_filename}")
print(f"Final dataset shape: {df.shape}")

--- 5. SAVING ML-READY DATASET ---
Success! Final ML-Ready dataset saved to CICIDS2017_ML_Ready_NoScale.csv
Final dataset shape: (2827876, 56)


In [20]:
print("--- 4. SCALING NUMERICAL FEATURES ---")
# Get the updated list of numeric columns after dropping the collinear ones
updated_numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print(f"Scaling {len(updated_numeric_cols)} numeric features using StandardScaler...")
scaler = StandardScaler()

# Fit and transform the numeric columns
df[updated_numeric_cols] = scaler.fit_transform(df[updated_numeric_cols])
print("Scaling complete. All numeric features now have mean=0 and variance=1.\n")

print("--- 5. SAVING ML-READY DATASET ---")
output_filename = 'CICIDS2017_ML_Ready.csv'
df.to_csv(output_filename, index=False)

print(f"Success! Final ML-Ready dataset saved to {output_filename}")
print(f"Final dataset shape: {df.shape}")

--- 4. SCALING NUMERICAL FEATURES ---
Scaling 53 numeric features using StandardScaler...
Scaling complete. All numeric features now have mean=0 and variance=1.

--- 5. SAVING ML-READY DATASET ---
Success! Final ML-Ready dataset saved to CICIDS2017_ML_Ready.csv
Final dataset shape: (2827876, 56)


In [30]:
try:
    del df
    print("Old dataframes cleared.")
except NameError:
    pass
gc.collect()

Old dataframes cleared.


0

In [33]:
import os
import seaborn as sns
import matplotlib.pyplot as plt
output_dir = 'data_analysis'
os.makedirs(output_dir, exist_ok=True)

In [ ]:
file_path = 'CICIDS2017_ML_Ready_NoScale.csv'
print(f"--- 2. LOADING {file_path} ---")
df = pd.read_csv(file_path, low_memory=False)

In [34]:
# 3. Create Binary_Label for the hue (coloring)
df['Binary_Label'] = df['Label'].apply(lambda x: 'Benign' if x == 'BENIGN' else 'Attack')

# 4. Select relevant, non-collinear features
# Note: 'Total Backward Packets' was dropped during preprocessing, so we use 'Total Fwd Packets'
# which now acts as the representative feature for both.
key_features = [
    'Flow Duration',       # Core temporal feature
    'Total Fwd Packets',   # Core volume feature
    'Flow IAT Mean',       # Core interval/timing feature
    'Flow Bytes/s'         # Core speed/throughput feature
]

# Ensure the features exist (safety check)
valid_features = [col for col in key_features if col in df.columns]

print(f"--- 3. GENERATING PAIRPLOT ---")
print(f"Selected relevant features: {valid_features}")
print("Sampling 100k rows (stratified) to ensure memory safety...")

# 5. Robust Stratified Sampling (100k total)
benign_df = df[df['Binary_Label'] == 'Benign']
attack_df = df[df['Binary_Label'] == 'Attack']

benign_sample = benign_df.sample(n=min(50000, len(benign_df)), random_state=42)
attack_sample = attack_df.sample(n=min(50000, len(attack_df)), random_state=42)

sample_df = pd.concat([benign_sample, attack_sample])

# 6. Generate and save the Pairplot
pairplot_fig = sns.pairplot(sample_df[valid_features + ['Binary_Label']], 
                            hue='Binary_Label', 
                            palette=['#2ecc71', '#e74c3c'], 
                            plot_kws={'alpha': 0.5, 's': 10}) 

pairplot_fig.fig.suptitle('Pairplot Density of Scaled ML-Ready Features (100k Sample)', y=1.02)

output_img = f'{output_dir}/06_ml_ready_pairplot_subset.png'
pairplot_fig.savefig(output_img, dpi=300, bbox_inches='tight')
plt.close()

# Cleanup
df.drop(columns=['Binary_Label'], inplace=True, errors='ignore')
print(f"\nSuccess! Scaled pairplot saved to {output_img}")

--- 3. GENERATING PAIRPLOT ---
Selected relevant features: ['Flow Duration', 'Total Fwd Packets', 'Flow IAT Mean', 'Flow Bytes/s']
Sampling 100k rows (stratified) to ensure memory safety...

Success! Scaled pairplot saved to data_analysis/06_ml_ready_pairplot_subset.png
